# Exploración del Dataset Task01_BrainTumour (BraTS)

Este notebook realiza un análisis exploratorio del dataset BraTS para comprender la distribución de las secuencias de resonancia magnética (MRI) y los mapas de segmentación tumoral. Inspirado en el artículo *"Deep Learning–based Identification of Brain MRI Sequences"* (2023), este notebook incluye:
1. Inspección de dimensiones y modalidades (FLAIR, T1w, T1gd, T2w).
2. Visualización de ejemplos multimodales con superposición de etiquetas.
3. Análisis de la distribución de clases en los volúmenes.
4. Análisis del área/volumen de los diferentes componentes del tumor.

### 1. Carga de dependencias e inspección de rutas

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Configurar estética visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

# Rutas del dataset
BASE_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "data", "Task01_BrainTumour")
images_dir = os.path.join(DATA_DIR, "imagesTr")
labels_dir = os.path.join(DATA_DIR, "labelsTr")

image_paths = sorted(glob.glob(os.path.join(images_dir, "*.nii.gz")))
label_paths = sorted(glob.glob(os.path.join(labels_dir, "*.nii.gz")))

print(f"Total de pacientes de entrenamiento: {len(image_paths)}")
print(f"Total de etiquetas de entrenamiento: {len(label_paths)}")

### 2. Visualización Multimodal de un Paciente

Cada volumen BraTS contiene 4 modalidades de MRI alineadas en el mismo espacio tridimensional:
- **Canal 0: FLAIR** (ideal para ver edema peritumoral)
- **Canal 1: T1w** (T1 ponderado simple)
- **Canal 2: T1gd / T1w-CE** (T1 ponderado con contraste de gadolinio, ideal para el núcleo de tumor realzado)
- **Canal 3: T2w** (T2 ponderado)

Las etiquetas representan:
- **0**: Fondo / Tejido sano
- **1**: Edema peritumoral (Edema)
- **2**: Núcleo tumoral no realzado (Necrotic/Non-enhancing tumor core)
- **3**: Tumor realzado (Enhancing tumor core)

In [ ]:
# Seleccionar un paciente de ejemplo
sample_idx = 0
img_nii = nib.load(image_paths[sample_idx])
lbl_nii = nib.load(label_paths[sample_idx])

img_data = img_nii.get_fdata()
lbl_data = lbl_nii.get_fdata()

print(f"Dimensiones de la imagen: {img_data.shape} (H, W, D, Modalities)")
print(f"Dimensiones de la etiqueta: {lbl_data.shape} (H, W, D)")

# Encontrar un corte central que contenga tumor realzado (Clase 3)
z_slices_with_tumor = np.where(np.any(lbl_data == 3, axis=(0, 1)))[0]
if len(z_slices_with_tumor) > 0:
    z_slice = z_slices_with_tumor[len(z_slices_with_tumor) // 2]
else:
    z_slice = img_data.shape[2] // 2

print(f"Visualizando corte axial z = {z_slice}")

# Visualizar las 4 secuencias en un grid
modalities = ["FLAIR", "T1w", "T1gd (Contraste)", "T2w"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, name in enumerate(modalities):
    axes[idx].imshow(img_data[:, :, z_slice, idx], cmap="gray")
    axes[idx].set_title(name)
    axes[idx].axis("off")
plt.suptitle(f"Secuencias MRI del Paciente {os.path.basename(image_paths[sample_idx])}", y=1.05, fontsize=16)
plt.tight_layout()
plt.show()

### 3. Superposición de la Segmentación Tumoral
Veamos los diferentes compartimentos tumorales superpuestos sobre las secuencias FLAIR y T1gd.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Corte FLAIR
axes[0].imshow(img_data[:, :, z_slice, 0], cmap="gray")
axes[0].set_title("MRI FLAIR (Fondo)")
axes[0].axis("off")

# Corte Etiqueta puro
im_lbl = axes[1].imshow(lbl_data[:, :, z_slice], cmap="jet", interpolation="nearest")
axes[1].set_title("Mapa de Segmentación")
axes[1].axis("off")

# Superposición
axes[2].imshow(img_data[:, :, z_slice, 0], cmap="gray")
masked_lbl = np.ma.masked_where(lbl_data[:, :, z_slice] == 0, lbl_data[:, :, z_slice])
axes[2].imshow(masked_lbl, cmap="jet", alpha=0.5, interpolation="nearest")
axes[2].set_title("Superposición (FLAIR + Tumor)")
axes[2].axis("off")

# Añadir barra de leyenda de clases
cbar = fig.colorbar(im_lbl, ax=axes, orientation='horizontal', shrink=0.5, pad=0.1)
cbar.set_ticks([0, 1, 2, 3])
cbar.set_ticklabels(["0: Sano", "1: Edema", "2: No-realzado", "3: Realzado"])

plt.suptitle(f"Compartimentos Tumorales (Corte {z_slice})", fontsize=16)
plt.show()

### 4. Distribución del Volumen Tumoral por Paciente
Vamos a calcular el tamaño (en número de vóxeles) de cada componente tumoral en una muestra de 20 pacientes para analizar la distribución e identificar si existe desbalance entre las subclases tumorales.

In [ ]:
stats = []
num_sample_patients = min(20, len(label_paths))

for i in range(num_sample_patients):
    lbl = nib.load(label_paths[i]).get_fdata()
    p_id = os.path.basename(label_paths[i]).replace(".nii.gz", "")
    
    # Contar vóxeles de cada clase
    counts = np.bincount(lbl.astype(int).ravel(), minlength=4)
    stats.append({
        "Paciente": p_id,
        "Edema (1)": counts[1],
        "No-realzado (2)": counts[2],
        "Realzado (3)": counts[3]
    })

df_stats = pd.DataFrame(stats)
print("Primeras filas de estadísticas de volumen:")
print(df_stats.head())

# Graficar boxplot de la distribución de tamaños de tumor
df_melt = df_stats.melt(id_vars="Paciente", var_name="Componente", value_name="Vóxeles")

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_melt, x="Componente", y="Vóxeles", palette="muted")
plt.title("Distribución de Vóxeles Tumorales en una Muestra de Pacientes")
plt.ylabel("Cantidad de Vóxeles")
plt.yscale("log") # Usar escala logarítmica debido al tamaño relativo
plt.show()

### Conclusiones de la Exploración
- **Desbalance de Clases**: El tejido sano (Fondo) ocupa más del 98% del volumen total. Entre las subclases tumorales, el edema (Clase 1) suele ser el más grande, seguido por el tumor realzado (Clase 3) y el tumor no realzado (Clase 2).
- **Secuencias representativas**: Para una clasificación 2D de ResNet-18, la combinación de canales (FLAIR, T1gd, T2w) captura la información más útil de los tres compartimentos para alimentar la red convolucional.